In [18]:
import os
extract_dir = r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\NetMHCpan_train"
print(os.listdir(extract_dir))

['allelelist', 'c000_ba', 'c000_el', 'c001_ba', 'c001_el', 'c002_ba', 'c002_el', 'c003_ba', 'c003_el', 'c004_ba', 'c004_el', 'MHC_pseudo.dat']


In [19]:
pseudo_path = r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\NetMHCpan_train\MHC_pseudo.dat"
with open(pseudo_path) as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i > 4:
            break

'BoLA-100901 YYSMYREISENVYGSNLYLLYRDYTWEYLNYRWY\n'
'BoLA-100902 YYSEYREISENVYESNLYLLYRDYTWEYLNYRWY\n'
'BoLA-101901 YHTKYREISENVYGSNLYYDYDYYTWAVFNYRGY\n'
'BoLA-102001 YHTKYREISENVYGSNLYFLYMDYTWAVFNYRGY\n'
'BoLA-102101 YYTKYREISENVYGSNLYFQFRYYTWADFNYEGY\n'
'BoLA-102301 YYSEYREISENVYESNLYIAYSDYTWEYLNYRWY\n'


In [20]:
pseudo_path = r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\NetMHCpan_train\MHC_pseudo.dat"

with open(pseudo_path) as f:
    hla_lines = [line for line in f if line.startswith('HLA-')]

print(f"Total HLA entries: {len(hla_lines)}")
print("First 10 HLA entries:")
for line in hla_lines[:10]:
    print(repr(line))

Total HLA entries: 10194
First 10 HLA entries:
'HLA-A0101 YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY\n'
'HLA-A0102 YSAMYQENMAHTDANTLYIIYRDYTWVARVYRGY\n'
'HLA-A0103 YFAMYQENMAHTDANTLYIMYRDYTWVARVYRGY\n'
'HLA-A0104 YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY\n'
'HLA-A0106 YFAMYQENMAHTDANTLYIIYRDYTWVALAYRGY\n'
'HLA-A0107 YFAMYQENVAHTDENTLYIIYRDYTWVARVYRGY\n'
'HLA-A0108 YFAMYQENMAHTDANTLYIIYRDYTWVARVYWGY\n'
'HLA-A0109 YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY\n'
'HLA-A0110 YFAMYQENMAHTDANTLYIIYRDYTWARRVYRGY\n'
'HLA-A0111 YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY\n'


In [21]:
import re

def normalize_allele(allele):
    """Convert 'HLA-A*02:01' -> 'HLA-A0201' to match MHC_pseudo.dat format"""
    return re.sub(r'[\*:]', '', allele)

# Test it
print(normalize_allele('HLA-A*02:01'))  # should print 'HLA-A0201'
print(normalize_allele('HLA-B*07:02'))  # should print 'HLA-B0702'

HLA-A0201
HLA-B0702


In [22]:
print(os.getcwd())  # Print the current working directory
print(os.path.exists("mhc_9mer_panallele_training_data.csv"))
print(os.path.exists("pseudo_sequences.csv"))

c:\Users\jbosb\OneDrive\Desktop\Python_Work\Hopkins Studies\Applied_ML\Personal projects
True
True


In [23]:
import pandas as pd

# Load your original bdata file
df = pd.read_csv(r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\bdata.20130222.mhci.txt", sep="\t")

df = df[df['peptide_length'] == 9]
df = df[df['inequality'] == '=']
df = df[df['mhc'].str.startswith('HLA-')]

standard_aas = set('ACDEFGHIKLMNPQRSTVWY')
df = df[df['sequence'].apply(lambda s: set(s).issubset(standard_aas))]

df = df.rename(columns={'sequence': 'peptide', 'mhc': 'allele', 'meas': 'ic50'})
df = df[['peptide', 'allele', 'ic50']].reset_index(drop=True)

# NEW: normalize allele names to match MHC_pseudo.dat format
df['allele_normalized'] = df['allele'].apply(normalize_allele)

# Load pseudosequence file
pseudo_path = r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\NetMHCpan_train\MHC_pseudo.dat"
pseudo_df = pd.read_csv(pseudo_path, sep=r'\s+', header=None, names=['allele', 'pseudoseq'])
pseudo_dict = dict(zip(pseudo_df['allele'], pseudo_df['pseudoseq']))

# Check overlap BEFORE filtering — sanity check
matched = df['allele_normalized'].isin(pseudo_dict.keys())
print(f"Rows with matching pseudosequence: {matched.sum()} / {len(df)}")
print(f"Unique alleles matched: {df.loc[matched, 'allele_normalized'].nunique()}")
print(f"Unique alleles NOT matched: {df.loc[~matched, 'allele_normalized'].nunique()}")
print("Sample unmatched alleles:", df.loc[~matched, 'allele'].unique()[:10])

# Filter to matched rows only, use the normalized name as the actual allele column going forward
df = df[matched].copy()
df['allele'] = df['allele_normalized']
df = df.drop(columns=['allele_normalized'])

print(f"\nFinal shape: {df.shape}")
print(f"Final unique alleles: {df['allele'].nunique()}")

df.to_csv(r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\mhc_9mer_panallele_training_data.csv", index=False)

# Also save pseudo_df for the training notebook to load directly
pseudo_df.to_csv(r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\pseudo_sequences.csv", index=False)

Rows with matching pseudosequence: 58822 / 58957
Unique alleles matched: 101
Unique alleles NOT matched: 13
Sample unmatched alleles: <StringArray>
[   'HLA-A1',   'HLA-A11',    'HLA-A2',   'HLA-A24',   'HLA-A26',    'HLA-A3',
 'HLA-A3/11',   'HLA-B51',   'HLA-B60',    'HLA-B7']
Length: 10, dtype: str

Final shape: (58822, 3)
Final unique alleles: 101


In [24]:
# %%
import pandas as pd

df = pd.read_csv(r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\mhc_9mer_panallele_training_data.csv")
pseudo_df = pd.read_csv(r"C:\Users\jbosb\OneDrive\Desktop\Python_Work\Personal projects\pseudo_sequences.csv")
pseudo_dict = dict(zip(pseudo_df['allele'], pseudo_df['pseudoseq']))

print("Data shape:", df.shape)
print("Unique alleles:", df['allele'].nunique())
print(df['allele'].value_counts().head(10))
print("\nIC50 value range:")
print(df['ic50'].describe())   # raw IC50, not a pre-transformed 'target' column

missing = set(df['allele'].unique()) - set(pseudo_dict.keys())
print(f"\n{len(missing)} alleles missing pseudosequence (should be 0)")

pseudo_lengths = pseudo_df['pseudoseq'].str.len().unique()
print("Pseudosequence lengths present:", pseudo_lengths)
assert len(pseudo_lengths) == 1, "Pseudosequences have inconsistent lengths"
PSEUDO_LEN = int(pseudo_lengths[0])


Data shape: (58822, 3)
Unique alleles: 101
allele
HLA-A0201    6038
HLA-A0301    2844
HLA-A0206    2777
HLA-A1101    2648
HLA-A0203    2631
HLA-A6802    2200
HLA-A3101    2190
HLA-B1501    2090
HLA-A0202    2055
HLA-B0702    1957
Name: count, dtype: int64

IC50 value range:
count    5.882200e+04
mean     1.239759e+04
std      9.415274e+04
min      1.000000e+00
25%      3.923774e+01
50%      5.439715e+02
75%      6.822080e+03
max      1.427660e+07
Name: ic50, dtype: float64

0 alleles missing pseudosequence (should be 0)
Pseudosequence lengths present: [34]


In [25]:

# GPU setup
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f'CUDNN Version: {torch.backends.cudnn.version()}')
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


Using device: cuda
CUDNN Version: 91002
GPU Name: NVIDIA GeForce RTX 4070 SUPER


In [26]:
# %%
# BLOSUM50 encoding setup — unchanged, but bl50_encode now needs a variable length
# since we're encoding both 9-mers and pseudosequences with the same function
from Bio.Align import substitution_matrices
import numpy as np

blossum50 = substitution_matrices.load("BLOSUM50")
standard_amino_acids = 'ARNDCQEGHILKMFPSTWYV'

bl50_lookup = {}
for aa in standard_amino_acids:
    bl50_lookup[aa] = np.array([blossum50[aa, other_aa] for other_aa in standard_amino_acids], dtype=np.float32)

all_scores = np.array(list(bl50_lookup.values()))
mean = np.mean(all_scores)
std = np.std(all_scores)

bl50_dict_normalized = {
    aa: (scores - mean) / std for aa, scores in bl50_lookup.items()
}

def bl50_encode(sequence, bl50_dict=bl50_dict_normalized, length=None):
    if length is None:
        length = len(sequence)
    arr = np.zeros((length, 20), dtype=np.float32)
    for i, aa in enumerate(sequence):
        if aa in bl50_dict:
            arr[i] = bl50_dict[aa]
        else:
            raise ValueError(f"Invalid amino acid '{aa}' found in sequence '{sequence}'.")
    return arr

# NEW: encode a peptide + its paired allele pseudosequence as one concatenated array
def encode_pair(peptide, allele, bl50_dict=bl50_dict_normalized, pep_len=9, pseudo_len=PSEUDO_LEN):
    pep_arr = bl50_encode(peptide, bl50_dict, length=pep_len)
    pseudo_seq = pseudo_dict[allele]
    pseudo_arr = bl50_encode(pseudo_seq, bl50_dict, length=pseudo_len)
    return np.concatenate([pep_arr, pseudo_arr], axis=0)  # (pep_len + pseudo_len, 20)


In [27]:
MIN_EXAMPLES_PER_ALLELE = 30

allele_counts = df['allele'].value_counts()
keep_alleles = allele_counts[allele_counts >= MIN_EXAMPLES_PER_ALLELE].index

print(f"Alleles before filtering: {df['allele'].nunique()}")
df = df[df['allele'].isin(keep_alleles)].reset_index(drop=True)
print(f"Alleles after filtering: {df['allele'].nunique()}")
print(f"Rows after filtering: {len(df)}")

Alleles before filtering: 101
Alleles after filtering: 83
Rows after filtering: 58607


In [28]:
classical_loci = ('HLA-A', 'HLA-B', 'HLA-C')

print("Locus breakdown before filtering:")
print(df['allele'].str.extract(r'(HLA-[A-Z]+)')[0].value_counts())

df = df[df['allele'].str.startswith(classical_loci)].reset_index(drop=True)

print(f"\nRows after restricting to classical A/B/C: {len(df)}")
print(f"Alleles after restricting to classical A/B/C: {df['allele'].nunique()}")

Locus breakdown before filtering:
0
HLA-A    39137
HLA-B    17722
HLA-C     1616
HLA-E      132
Name: count, dtype: int64

Rows after restricting to classical A/B/C: 58475
Alleles after restricting to classical A/B/C: 81


In [29]:
# Build the encoded dataset — this is the main change from single-allele version:
# each row is now peptide + allele pseudosequence, not peptide alone
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import WeightedRandomSampler

X = np.stack([
    encode_pair(row.peptide, row.allele)
    for row in df.itertuples()
])# shape: (N, 9 + PSEUDO_LEN, 20)

# Target: NetMHCpan's target column is already on the 0-1 transformed scale,
# so we use it directly — do NOT re-apply 1 - log(ic50)/log(50000) here.
y = 1 - np.log(df['ic50'].values.astype(np.float32)) / np.log(50000)
y = np.clip(y, 0, 1).astype(np.float32)

X_tensor = torch.tensor(X)
y_tensor = torch.tensor(y)

print("X shape:", X_tensor.shape)   # e.g. (N, 43, 20)
print("y shape:", y_tensor.shape)

X shape: torch.Size([58475, 43, 20])
y shape: torch.Size([58475])


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val, allele_train, allele_val = train_test_split(
    X_tensor, y_tensor, df['allele'].values, test_size=0.2, random_state=42,
    stratify=df['allele'].values
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_ds, batch_size=64)

# Balanced sampler — unchanged
from torch.utils.data import WeightedRandomSampler

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)

CORE_MIN_EXAMPLES = 200   # tune this — should meaningfully exceed your rare-allele cutoff
train_allele_counts = pd.Series(allele_train).value_counts()
core_alleles = train_allele_counts[train_allele_counts >= CORE_MIN_EXAMPLES].index
print(f"Core alleles (burn-in): {len(core_alleles)}")
print(core_alleles.tolist())

core_mask = pd.Series(allele_train).isin(core_alleles).values
X_train_core = X_train[core_mask]
y_train_core = y_train[core_mask]
print(f"Core burn-in training rows: {len(X_train_core)} / {len(X_train)} total")

# --- Burn-in loader: core alleles only, same affinity-bin balancing as before ---
train_ds_core = TensorDataset(X_train_core, y_train_core)

bin_idx_core = np.digitize(y_train_core.numpy(), bin_edges[1:-1], right=True)
bin_counts_core = np.clip(np.bincount(bin_idx_core, minlength=n_bins), 1, None)
weights_core = 1.0 / bin_counts_core[bin_idx_core]
sampler_core = WeightedRandomSampler(weights_core, num_samples=len(weights_core), replacement=True)

train_loader_core = DataLoader(train_ds_core, batch_size=64, sampler=sampler_core)

# --- Full loader: everything, ideally with the allele-balanced sampler from before ---
group_key_full = pd.Series(allele_train).astype(str) + "_" + pd.Series(
    np.digitize(y_train.numpy(), bin_edges[1:-1], right=True)
).astype(str)
group_counts_full = group_key_full.value_counts()
weights_full = group_key_full.map(lambda g: 1.0 / group_counts_full[g]).values

sampler_full = WeightedRandomSampler(weights_full, num_samples=len(weights_full), replacement=True)
train_loader_full = DataLoader(train_ds, batch_size=64, sampler=sampler_full)

Train size: 46780, Val size: 11695
Core alleles (burn-in): 42
['HLA-A0201', 'HLA-A0301', 'HLA-A0206', 'HLA-A1101', 'HLA-A0203', 'HLA-A6802', 'HLA-A3101', 'HLA-B1501', 'HLA-A0202', 'HLA-B0702', 'HLA-A2402', 'HLA-A6801', 'HLA-A0101', 'HLA-B0801', 'HLA-A3301', 'HLA-B3501', 'HLA-A3001', 'HLA-A2902', 'HLA-A2601', 'HLA-B5801', 'HLA-A2301', 'HLA-B4001', 'HLA-A3002', 'HLA-B2705', 'HLA-B5101', 'HLA-B5701', 'HLA-B5301', 'HLA-A3201', 'HLA-B1801', 'HLA-B4402', 'HLA-B4403', 'HLA-A6901', 'HLA-B1517', 'HLA-B5401', 'HLA-B1503', 'HLA-A2403', 'HLA-B3901', 'HLA-A0211', 'HLA-B4002', 'HLA-A0212', 'HLA-B4501', 'HLA-A0219']
Core burn-in training rows: 43110 / 46780 total


C:\Users\jbosb\AppData\Local\Temp\ipykernel_62380\300339984.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  X_train_core = X_train[core_mask]


In [31]:
# Model — unchanged class, just needs the new (larger) input_dim
import torch.nn as nn

INPUT_DIM = X_tensor.shape[1] * X_tensor.shape[2]  # (9 + PSEUDO_LEN) * 20
print("Model input_dim:", INPUT_DIM)

class MHCNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x.flatten(1))

from scipy.stats import pearsonr
import copy

def training_eval_staged(hidden_size, burn_in_epochs=20, full_epochs=180, lr=0.001, patience=15):
    model = MHCNN(input_dim=INPUT_DIM, hidden=hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    epochs_without_improvement = 0
    total_epoch = 0

    # ---- Phase 1: burn-in on core alleles only ----
    for epoch in range(burn_in_epochs):
        model.train()
        for X_batch, y_batch in train_loader_core:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()
        total_epoch += 1

    print(f"  hidden={hidden_size}: burn-in complete ({burn_in_epochs} epochs on core alleles)")

    # ---- Phase 2: full training on all alleles, with early stopping ----
    for epoch in range(full_epochs):
        model.train()
        for X_batch, y_batch in train_loader_full:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()
        total_epoch += 1

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += loss_fn(model(X_batch).squeeze(), y_batch).item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"  hidden={hidden_size}: early stopping in phase 2 at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch).squeeze()
            val_preds.append(y_pred.cpu().numpy())
            val_true.append(y_batch.cpu().numpy())

    val_preds = np.concatenate(val_preds)
    val_true = np.concatenate(val_true)
    pcc = pearsonr(val_preds, val_true)[0]
    print(f"hidden={hidden_size}, Best Val Loss: {best_val_loss:.4f}, Val PCC: {pcc:.4f}, total_epochs={total_epoch}")

    return model, pcc

Model input_dim: 860


In [32]:
# Hidden-size sweep — unchanged logic, now operating on the pan-allele model
hidden_sizes = [2, 4, 6, 8, 10]  # widened slightly since input is much larger now
results = {}
models = {}

for hidden in hidden_sizes:
    model, pcc = training_eval_staged(hidden)
    results[hidden] = pcc
    models[hidden] = model

best_hidden = max(results, key=results.get)
print(f"\nBest hidden size: {best_hidden} (PCC = {results[best_hidden]:.4f})")

# %%
# NEW: per-allele PCC breakdown — aggregate PCC alone can hide poor performance
# on underrepresented alleles, so check this before trusting the aggregate number
best_model = models[best_hidden]
best_model.eval()

with torch.no_grad():
    preds = best_model(X_val.to(device)).squeeze().cpu().numpy()

val_df = pd.DataFrame({
    'allele': allele_val,
    'pred': preds,
    'true': y_val.numpy()
})

per_allele_pcc = val_df.groupby('allele').apply(
    lambda g: pearsonr(g['pred'], g['true'])[0] if len(g) > 5 else np.nan
).sort_values(ascending=False)

print("Per-allele Val PCC (alleles with >5 val examples):")
print(per_allele_pcc)
print(f"\nMean per-allele PCC: {per_allele_pcc.mean():.4f}")
print(f"Number of alleles with usable val PCC: {per_allele_pcc.notna().sum()}")

  hidden=2: burn-in complete (20 epochs on core alleles)
  hidden=2: early stopping in phase 2 at epoch 39
hidden=2, Best Val Loss: 0.0773, Val PCC: 0.3870, total_epochs=59
  hidden=4: burn-in complete (20 epochs on core alleles)
  hidden=4: early stopping in phase 2 at epoch 40
hidden=4, Best Val Loss: 0.0664, Val PCC: 0.4765, total_epochs=60
  hidden=6: burn-in complete (20 epochs on core alleles)
  hidden=6: early stopping in phase 2 at epoch 30
hidden=6, Best Val Loss: 0.0610, Val PCC: 0.5482, total_epochs=50
  hidden=8: burn-in complete (20 epochs on core alleles)
  hidden=8: early stopping in phase 2 at epoch 41
hidden=8, Best Val Loss: 0.0500, Val PCC: 0.6448, total_epochs=61
  hidden=10: burn-in complete (20 epochs on core alleles)
  hidden=10: early stopping in phase 2 at epoch 25
hidden=10, Best Val Loss: 0.0531, Val PCC: 0.6252, total_epochs=45

Best hidden size: 8 (PCC = 0.6448)
Per-allele Val PCC (alleles with >5 val examples):
allele
HLA-A0205    0.979144
HLA-B4201    0.8

In [33]:
allele_val_counts = pd.Series(allele_val).value_counts()
diagnostic = per_allele_pcc.to_frame('pcc').join(allele_val_counts.rename('n_val'))
print(diagnostic.sort_values('pcc').head(15))
print("\n---")
print(diagnostic.sort_values('pcc', ascending=False).head(15))

                pcc  n_val
allele                    
HLA-B4013 -0.162803     12
HLA-B2720 -0.128127     18
HLA-C0501 -0.085116     20
HLA-B1401 -0.018059      8
HLA-A2501  0.020516     16
HLA-A0250  0.059951     18
HLA-B5802  0.065963     10
HLA-B5703  0.074663      6
HLA-C0401  0.107846     35
HLA-B1502  0.138444     28
HLA-C1203  0.194254     34
HLA-B0803  0.208967      6
HLA-C0701  0.231266     43
HLA-A3215  0.267237     15
HLA-A8001  0.290999     37

---
                pcc  n_val
allele                    
HLA-A0205  0.979144      7
HLA-B4201  0.889445     19
HLA-A0216  0.839655     45
HLA-A0212  0.810644     76
HLA-C0602  0.800176     36
HLA-A0219  0.788523     56
HLA-A0201  0.778669   1208
HLA-B3801  0.740942     44
HLA-A2403  0.733587     91
HLA-A0202  0.732721    411
HLA-A0203  0.719244    526
HLA-A0211  0.716893     85
HLA-B0802  0.708354     21
HLA-B5801  0.700208    203
HLA-B4002  0.691676     80
